[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/OL7014/blob/main/notebooks/week5_transportation.ipynb)

In [ ]:
%pip install -q gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB

params = {
    "WLSACCESSID": "paste your ACCESSID here",
    "WLSSECRET":   "paste your SECRET here",
    "LICENSEID":   123456,   # the number itself, no quotes
}
env = gp.Env(params=params)

# Transportation — the game's board

Two depots ship crates to three divisions. Every depot can reach every division, and crates ship whole $\Rightarrow x_{ij} \in \mathbb{Z}_+$.

|  | Div 1 | Div 2 | Div 3 | supply |
| :--- | ---: | ---: | ---: | ---: |
| Depot A | 4 | 5 | 9 | **5** |
| Depot B | 3 | 6 | 6 | **7** |
| demand | **6** | **2** | **4** | 12 = 12 |

Inner cells = cost per crate $c_{ij}$. $\;x_{A1}$ = crates shipped from depot A to division 1 (the note's $x_{11}$).

$$
\begin{aligned}
\min_{x} \quad & 4x_{A1} + 5x_{A2} + 9x_{A3} + 3x_{B1} + 6x_{B2} + 6x_{B3} \\
\text{s.t.} \quad & x_{A1} + x_{A2} + x_{A3} \le 5 \qquad \text{depot A ships no more than it holds} \\
                  & x_{B1} + x_{B2} + x_{B3} \le 7 \qquad \text{depot B} \\
                  & x_{A1} + x_{B1} \ge 6 \qquad \text{division 1 gets what it needs} \\
                  & x_{A2} + x_{B2} \ge 2 \qquad \text{division 2} \\
                  & x_{A3} + x_{B3} \ge 4 \qquad \text{division 3} \\
                  & x_{A1},\, x_{A2},\, x_{A3},\, x_{B1},\, x_{B2},\, x_{B3} \in \mathbb{Z}_+
\end{aligned}
$$

**Math to code**

| math | code |
| :--- | :--- |
| $x_{ij} \in \mathbb{Z}_+$ | `addVar(vtype=GRB.INTEGER)` — lower bound is already `0` |
| $\min$ total cost | `setObjective(expr, GRB.MINIMIZE)` — expression **and** sense |
| each $\le$ / $\ge$ row | one `addConstr` — same coefficients, same order |

## Live coding — fill in the blanks

Skeleton for building the model live in class, one comment at a time.

In [ ]:
# Initialize the model gp.Model()


# decision variables, six integer variables x_A1, ..., x_B3 --- e.g., ip.addVar(vtype=GRB.INTEGER)


# let's define objective function, total shipping cost --- ip.setObjective(..., GRB.MINIMIZE)


# supply constraints, one per depot --- ip.addConstr(... <= ...)


# demand constraints, one per division --- ip.addConstr(... >= ...)


# Model is set up. Let's optimize --- ip.optimize()


# Let's print the six shipments and the total cost

## Reference: the completed model

By hand — no dictionaries, no loops, no `quicksum` — so every term matches the math above.

In [ ]:
ip = gp.Model("transportation_ip")

# decision variables -- crates on each road, whole numbers, lower bound 0 by default
x_A1 = ip.addVar(vtype=GRB.INTEGER, name="x_A1")
x_A2 = ip.addVar(vtype=GRB.INTEGER, name="x_A2")
x_A3 = ip.addVar(vtype=GRB.INTEGER, name="x_A3")
x_B1 = ip.addVar(vtype=GRB.INTEGER, name="x_B1")
x_B2 = ip.addVar(vtype=GRB.INTEGER, name="x_B2")
x_B3 = ip.addVar(vtype=GRB.INTEGER, name="x_B3")

# objective -- total shipping cost, written out term by term
ip.setObjective(4 * x_A1 + 5 * x_A2 + 9 * x_A3 + 3 * x_B1 + 6 * x_B2 + 6 * x_B3, GRB.MINIMIZE)

# supply -- each depot ships no more than it holds
ip.addConstr(x_A1 + x_A2 + x_A3 <= 5, name="supply_A")
ip.addConstr(x_B1 + x_B2 + x_B3 <= 7, name="supply_B")

# demand -- each division gets what it needs
ip.addConstr(x_A1 + x_B1 >= 6, name="demand_1")
ip.addConstr(x_A2 + x_B2 >= 2, name="demand_2")
ip.addConstr(x_A3 + x_B3 >= 4, name="demand_3")

ip.optimize()

print(f"\nA -> 1: {round(x_A1.X)}   A -> 2: {round(x_A2.X)}   A -> 3: {round(x_A3.X)}")
print(f"B -> 1: {round(x_B1.X)}   B -> 2: {round(x_B2.X)}   B -> 3: {round(x_B3.X)}")
print(f"total cost = {ip.ObjVal:.0f}")

## Now relax — delete the integrality line

Same model, one keyword different: `CONTINUOUS` instead of `INTEGER`.

On week 1's toy this step moved the answer from $(1,\,3)$ to $(2.5,\; 2.5)$. Watch what it does here.

In [ ]:
lp = gp.Model("transportation_lp")

# same six roads -- CONTINUOUS now: any amount >= 0, fractions allowed
r_A1 = lp.addVar(vtype=GRB.CONTINUOUS, name="r_A1")
r_A2 = lp.addVar(vtype=GRB.CONTINUOUS, name="r_A2")
r_A3 = lp.addVar(vtype=GRB.CONTINUOUS, name="r_A3")
r_B1 = lp.addVar(vtype=GRB.CONTINUOUS, name="r_B1")
r_B2 = lp.addVar(vtype=GRB.CONTINUOUS, name="r_B2")
r_B3 = lp.addVar(vtype=GRB.CONTINUOUS, name="r_B3")

lp.setObjective(4 * r_A1 + 5 * r_A2 + 9 * r_A3 + 3 * r_B1 + 6 * r_B2 + 6 * r_B3, GRB.MINIMIZE)
lp.addConstr(r_A1 + r_A2 + r_A3 <= 5, name="supply_A")
lp.addConstr(r_B1 + r_B2 + r_B3 <= 7, name="supply_B")
lp.addConstr(r_A1 + r_B1 >= 6, name="demand_1")
lp.addConstr(r_A2 + r_B2 >= 2, name="demand_2")
lp.addConstr(r_A3 + r_B3 >= 4, name="demand_3")

lp.optimize()

print(f"\n{'road':8}{'IP':>6}{'LP':>9}")
for road, xi, xr in [("A -> 1", x_A1, r_A1), ("A -> 2", x_A2, r_A2), ("A -> 3", x_A3, r_A3),
                     ("B -> 1", x_B1, r_B1), ("B -> 2", x_B2, r_B2), ("B -> 3", x_B3, r_B3)]:
    print(f"{road:8}{round(xi.X):6d}{abs(xr.X):9.2f}")   # abs(): the solver may hand back -0.0
print(f"{'cost':8}{ip.ObjVal:6.0f}{lp.ObjVal:9.2f}")

## Plot both plans

One cell per road, number = crates shipped. Left: the IP. Right: the LP relaxation, which was allowed to answer $2.5$ and did not.

In [ ]:
import matplotlib.pyplot as plt

ip_plan = [[x_A1.X, x_A2.X, x_A3.X], [x_B1.X, x_B2.X, x_B3.X]]
lp_plan = [[r_A1.X, r_A2.X, r_A3.X], [r_B1.X, r_B2.X, r_B3.X]]

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, plan, title in [(axes[0], ip_plan, f"IP: cost = {ip.ObjVal:.0f}"),
                        (axes[1], lp_plan, f"LP relaxation: cost = {lp.ObjVal:.2f}")]:
    ax.imshow(plan, cmap="Blues", vmin=0, vmax=6)
    for i in range(2):
        for j in range(3):
            ax.text(j, i, f"{abs(plan[i][j]):g}", ha="center", va="center", fontsize=14)
    ax.set_xticks(range(3), ["Div 1", "Div 2", "Div 3"])
    ax.set_yticks(range(2), ["Depot A", "Depot B"])
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Why? Look at $A$

The solver sees one row per constraint, one column per road:

|  | $x_{A1}$ | $x_{A2}$ | $x_{A3}$ | $x_{B1}$ | $x_{B2}$ | $x_{B3}$ |  |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| supply A | 1 | 1 | 1 | 0 | 0 | 0 | $\le 5$ |
| supply B | 0 | 0 | 0 | 1 | 1 | 1 | $\le 7$ |
| demand 1 | 1 | 0 | 0 | 1 | 0 | 0 | $\ge 6$ |
| demand 2 | 0 | 1 | 0 | 0 | 1 | 0 | $\ge 2$ |
| demand 3 | 0 | 0 | 1 | 0 | 0 | 1 | $\ge 4$ |

Two blocks of rows (depots, divisions), and every column has one $1$ in each block — shape (b) of the note's Prop, so $A$ is **TU**. Every corner of the LP is a whole-number plan.

Check it by brute force: every square submatrix, every size.

In [ ]:
import itertools
import numpy as np

A = np.array([
    [1, 1, 1, 0, 0, 0],   # supply A
    [0, 0, 0, 1, 1, 1],   # supply B
    [1, 0, 0, 1, 0, 0],   # demand 1
    [0, 1, 0, 0, 1, 0],   # demand 2
    [0, 0, 1, 0, 0, 1],   # demand 3
])

def all_dets(M):
    m, n = M.shape
    found = set()
    for k in range(1, min(m, n) + 1):
        for rows in itertools.combinations(range(m), k):
            for cols in itertools.combinations(range(n), k):
                found.add(round(np.linalg.det(M[np.ix_(rows, cols)])))
    return sorted(found)

print("transportation A :", all_dets(A))                          # only -1, 0, 1 -> TU
print("week 1's toy A   :", all_dets(np.array([[1, 3], [3, 1]])))  # a 3 and a -8 -> not TU

## Any whole-number supplies and demands

TU is a property of $A$ alone — no $b$, no $c$. So change the supplies, demands and costs at random, keep them whole, and relax every time.

In [ ]:
import random

random.seed(5)
fractional = 0
for trial in range(200):
    total = random.randint(9, 20)
    a = random.randint(1, total - 1)
    supply = [a, total - a]                       # depot A, depot B
    cut = sorted(random.sample(range(1, total), 2))
    demand = [cut[0], cut[1] - cut[0], total - cut[1]]   # division 1, 2, 3
    cost = [[random.randint(1, 9) for j in range(3)] for i in range(2)]

    m = gp.Model()
    m.Params.OutputFlag = 0
    x = m.addVars(2, 3, vtype=GRB.CONTINUOUS)      # relaxed: fractions allowed
    m.setObjective(gp.quicksum(cost[i][j] * x[i, j] for i in range(2) for j in range(3)), GRB.MINIMIZE)
    m.addConstrs(x.sum(i, "*") <= supply[i] for i in range(2))
    m.addConstrs(x.sum("*", j) >= demand[j] for j in range(3))
    m.optimize()

    if any(abs(x[i, j].X - round(x[i, j].X)) > 1e-6 for i in range(2) for j in range(3)):
        fractional += 1

print(f"fractional LP answers: {fractional} / 200")

**Delete the integrality line and the answer does not move — for every whole-number supply and demand.** Not luck, not rounding: the reason is the shape of $A$.